### This notebook will prepare the datalist corresponding to the individual citation reports as done in the journal_classifier notebook for journals.

In [2]:
import pandas as pd
import numpy as np
import pickle

bindividuals = pd.read_excel('DATAFILES/individuals_sample.xlsx',sheet_name='researchers')
individuals = bindividuals.copy()
pickle.dump(individuals, open('HOME_PICKLE_FILES/individual_list.pkl','wb'))

In [4]:
individuals = pd.read_pickle('HOME_PICKLE_FILES/individual_list.pkl')
oobasic = pd.read_pickle("HOME_PICKLE_FILES/oonames.pkl")
instlist = pd.read_pickle('HOME_PICKLE_FILES/instlist.pkl')
instlista = instlist[['ooidentifier','incitesname']]

In [5]:
import glob
import os
os.chdir("INDIVIDUAL_FILES")
files = sorted(glob.glob('*.txt'))

In [6]:
datalist = []
researchers = []

In [7]:
for file in files:
    data = pd.read_csv(file, sep="\t",encoding='latin-1',low_memory=False)
    res = file.split('.')[0]
    researchers.append(res)
    if len(data.iloc[1, 0])>3:
        colnames = list(data.columns)
        colnames.append("final")
        data.columns = colnames[1:]
    data = data[['UT','SO','C1']]
    ndata = data['C1'].map(str)
    ndata = ndata.apply(lambda x: pd.Series(str(x).split("]")))
    ndata = ndata.apply(lambda x: x.astype(str).str.upper())
    sandata = pd.DataFrame(ndata)
    data = data[['UT','SO']]
    data = data.reset_index(drop=True)
    sdata = data.copy()
    todata = sdata.copy()
    for k in range(1,min(15,len(ndata.columns))):
        df = pd.DataFrame(sandata[k].str.split(',').str[0].str.strip())
        df = df.rename(columns={k:'oonames'})
        df = df.merge(oobasic, on='oonames',how='left')
        df = df.merge(instlista, on='ooidentifier',how='left')
        df = df.rename(columns={'incitesname':k})
        sdata = pd.concat([sdata, pd.DataFrame(df[k])], axis=1)
    sdata.insert(2, 'ninst', min(14, len(ndata.columns)-1) - sdata.isna().sum(axis=1))
    for k in range(1,min(15,len(ndata.columns))):
        df = sdata[['UT','SO','ninst',k]]
        df = df.rename(columns={k:'incitesname'})
        todata = pd.concat([todata,df],ignore_index=True)
    todata = todata.dropna()
    todata['ninst'] = todata['ninst'].astype(int)
    todata.sort_values('UT',ascending=True, inplace=True)
    todata = todata.reset_index(drop=True)
    todata = todata.rename(columns={'SO':'eco_incites'})
    todata = todata.rename(columns={'incitesname':'eco_institution'})
    todata = todata.reset_index(drop=True)
    todata = todata[['eco_incites','eco_institution','ninst']]
    datalist.append(todata) 

In [8]:
os.chdir("..")
pickle.dump(researchers, open('HOME_PICKLE_FILES/researchers_list.pkl','wb'))
pickle.dump(datalist, open('HOME_PICKLE_FILES/resdatalist.pkl','wb'))

In [9]:
print('The end')

The end
